# Donchian + VWAP validation (IS / OOS from 2023)

**Strategy (paper research):**
- **Universe:** S&P 500 ∪ Nasdaq-100, ex-pharma (same as LOOP)
- **Entry:** Close breaks **55-day** Donchian upper channel (prior bar channel) **and** close **>** **20-day rolling VWAP**
- **Exit:** Close **<** **20-day** Donchian lower channel, or drop at weekly rebalance if no longer eligible
- **Portfolio:** Top **8** by breakout strength, **equal weight**, max **2 / sector**, rebalance every **5** sessions
- **Signals:** Prior-session close (t−1) on rebalance days

**Split:**
| Sample | Period |
|--------|--------|
| **IS** | 2023-01-01 → 2024-12-31 |
| **OOS** | 2025-01-01 → 2026-09-26 |

Compare to **SPY** and **QQQ** on the same windows.


In [ ]:
import sys
from pathlib import Path
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.research.donchian_vwap_backtest import (
    PeriodMetrics,
    _metrics,
    benchmark_returns,
    is_oos_split,
    run_donchian_vwap_portfolio,
)

(is_start, is_end), (oos_start, oos_end) = is_oos_split()
full_start, full_end = is_start, oos_end
print(f"Full run: {full_start} → {full_end}")
print(f"IS: {is_start} → {is_end}")
print(f"OOS: {oos_start} → {oos_end}")


In [ ]:
# Full backtest (download once; ~503 symbols — may take several minutes)
daily_ret, eq = run_donchian_vwap_portfolio(full_start, full_end)
eq.tail()


In [ ]:
spy_r = benchmark_returns(full_start, full_end, "SPY")
qqq_r = benchmark_returns(full_start, full_end, "QQQ")

rows = []
for label, s, e in [
    ("Donchian+VWAP", full_start, full_end),
    ("Donchian+VWAP IS", is_start, is_end),
    ("Donchian+VWAP OOS", oos_start, oos_end),
    ("SPY full", full_start, full_end),
    ("SPY IS", is_start, is_end),
    ("SPY OOS", oos_start, oos_end),
    ("QQQ OOS", oos_start, oos_end),
]:
    r = daily_ret if label.startswith("Donchian") else (spy_r if "SPY" in label else qqq_r)
    ss, ee = (s, e)
    if "IS" in label:
        ss, ee = is_start, is_end
    elif "OOS" in label:
        ss, ee = oos_start, oos_end
    m = _metrics(label, ss, ee, r)
    rows.append(m.__dict__)

summary = pd.DataFrame(rows)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

def cum(ret):
    return (1 + ret.fillna(0)).cumprod() * 100

# Full path
axes[0].plot(cum(daily_ret), label="Donchian+VWAP")
axes[0].plot(cum(spy_r.reindex(daily_ret.index).fillna(0)), label="SPY", alpha=0.8)
axes[0].axvline(pd.Timestamp(is_end), color="gray", ls="--", label="IS/OOS split")
axes[0].set_title("Cumulative return index (100 = start)")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=30)

# OOS zoom
for r, lab in [(daily_ret, "Donchian+VWAP"), (spy_r, "SPY"), (qqq_r, "QQQ")]:
    sub = r[(r.index >= pd.Timestamp(oos_start)) & (r.index <= pd.Timestamp(oos_end))]
    axes[1].plot(cum(sub), label=lab)
axes[1].set_title("OOS window only")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
out = ROOT / "pipeline/research/artifacts/donchian_vwap_is_oos.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=144)
plt.show()
print("Saved", out)


## How to read results

- **IS (2023–2024):** In-sample — check whether the rules have *any* edge in a prior regime (post-2022 bear, 2023 recovery).
- **OOS (2025–2026):** Out-of-sample — no parameter changes; same 55/20/VWAP/top-8 rules. OOS should be **positive vs SPY** if the approach is robust (not guaranteed).

**Limits:** Daily rolling VWAP ≠ intraday session VWAP; this is an **EOD swing proxy**. No transaction costs or slippage.


In [ ]:
is_m = _metrics("IS", is_start, is_end, daily_ret)
oos_m = _metrics("OOS", oos_start, oos_end, daily_ret)
spy_is = _metrics("SPY IS", is_start, is_end, spy_r)
spy_oos = _metrics("SPY OOS", oos_start, oos_end, spy_r)

print(f"IS  Donchian+VWAP: {is_m.total_return_pct:+.2f}% (SPY {spy_is.total_return_pct:+.2f}%)")
print(f"OOS Donchian+VWAP: {oos_m.total_return_pct:+.2f}% (SPY {spy_oos.total_return_pct:+.2f}%)")
print(f"IS  Sharpe: {is_m.sharpe:.2f}  |  OOS Sharpe: {oos_m.sharpe:.2f}")
